# 02 — Star Detection

**Team:** StellarX  
**Phase:** 2 — Star Detection  
**Status:** ✅ Fully executable — Phase 2 implementation complete.

## What this notebook covers

1. Load configuration and catalog
2. Generate synthetic star-field images
3. Visualise each preprocessing step (raw → background subtracted → noise reduced → normalised)
4. Run star detection and overlay ground-truth vs. detected positions
5. Evaluate centroid accuracy against known ground-truth
6. Tune and compare detection threshold strategies
7. Analyse detection statistics across a larger batch
8. Phase 3 preparation notes

> **Note:** All images are **synthetic**, generated from the Hipparcos bright-star catalog
> using a simplified spacecraft star-sensor simulation.  They are not real imagery.

## Prerequisites

```bash
pip install -r requirements.txt
```

Run from the **repository root** or adjust the `sys.path` cell below.

In [ ]:
# Uncomment if running from inside notebooks/
# import sys; sys.path.insert(0, '..')

import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

warnings.filterwarnings('ignore')
matplotlib.use('Agg')
%matplotlib inline

with open('config.yaml') as f:
    config = yaml.safe_load(f)

ds_cfg  = config['dataset']
pp_cfg  = config['preprocessing']
det_cfg = config['star_detection']

print('Configuration loaded.')
print(f"  Background method : {pp_cfg['background_method']} (size={pp_cfg['background_filter_size']})")
print(f"  Noise method      : {pp_cfg['noise_method']} (sigma={pp_cfg['noise_sigma']})")
print(f"  Normalisation     : {pp_cfg['normalization']}")
print(f"  Threshold method  : {det_cfg['threshold_method']} (min={det_cfg['min_brightness']})")
print(f"  Max stars         : {det_cfg['max_stars']}")

---
## 1. Load catalog and generate sample images

In [ ]:
from src.catalog.catalog_loader import load_catalog
from src.preprocessing.star_field_generator import StarFieldGenerator

catalog = load_catalog(ds_cfg['catalog_file'], config)
print(catalog)

# Use 256x256 for notebook visualisation (faster than 512x512)
vis_cfg = {**ds_cfg, 'image_width': 256, 'image_height': 256,
           'shot_noise': False, 'artifact_probability': 0.0}
generator = StarFieldGenerator(catalog, vis_cfg)

# Find a seed that produces at least 2 visible stars
sample_sf = None
for seed in range(500):
    sf = generator.generate(seed=seed)
    if len(sf.stars) >= 2:
        sample_sf = sf
        print(f'Using seed={seed}  n_stars={len(sf.stars)}')
        break

if sample_sf is None:
    # Fallback: force a specific boresight close to Sirius
    sample_sf = generator.generate(seed=0, boresight_ra_deg=101.0, boresight_dec_deg=-16.0)
    print(f'Fallback: seed=0 boresight near Sirius  n_stars={len(sample_sf.stars)}')

---
## 2. Visualise each preprocessing step

In [ ]:
from src.preprocessing.image_preprocessing import (
    subtract_background, reduce_noise, normalise
)

raw   = sample_sf.image.copy()
bgsub = subtract_background(
    raw,
    method=pp_cfg['background_method'],
    filter_size=pp_cfg['background_filter_size']
)
denoised = reduce_noise(
    bgsub,
    method=pp_cfg['noise_method'],
    sigma=pp_cfg['noise_sigma']
)
normed = normalise(denoised, method=pp_cfg['normalization'])

stages = [
    (raw,      'Raw image'),
    (bgsub,    'Background subtracted'),
    (denoised, 'Noise reduced'),
    (normed,   'Normalised'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (img, title) in zip(axes, stages):
    # Log stretch for visibility
    disp = np.log1p(img * 50) / np.log1p(50)
    im = ax.imshow(disp, cmap='gray', vmin=0, vmax=1, origin='upper')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    stats = f'min={img.min():.4f}  max={img.max():.4f}  mean={img.mean():.4f}'
    ax.set_xlabel(stats, fontsize=7)

fig.suptitle('Preprocessing Pipeline — Synthetic Star-Field\n'
             f'[seed={sample_sf.seed}  n_gt_stars={len(sample_sf.stars)}]',
             fontsize=10)
plt.tight_layout()
plt.savefig('notebooks/fig_04_preprocessing_pipeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_04_preprocessing_pipeline.png')

---
## 3. Run star detection and compare with ground truth

In [ ]:
from src.preprocessing.star_detection import detect_stars

detections = detect_stars(normed, det_cfg)

print(f'Ground-truth stars : {len(sample_sf.stars)}')
print(f'Detected candidates: {len(detections)}')
print()
print('Ground truth:')
for s in sample_sf.stars:
    print(f'  {s.star_id:12s}  x={s.x_px:7.2f}  y={s.y_px:7.2f}  '
          f'flux={s.flux:.4f}  vmag={s.vmag:.2f}')

print()
print('Detections:')
for d in detections:
    print(f'  x={d.x:7.2f}  y={d.y:7.2f}  '
          f'brightness={d.brightness:.4f}  peak={d.peak:.4f}  area={d.area}')

In [ ]:
import math

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (img, label) in zip(axes, [(raw, 'Raw'), (normed, 'Preprocessed + Detected')]):
    disp = np.log1p(img * 50) / np.log1p(50)
    ax.imshow(disp, cmap='gray', vmin=0, vmax=1, origin='upper')

    # Ground-truth stars — green circles
    for s in sample_sf.stars:
        circ = plt.Circle((s.x_px, s.y_px), radius=6,
                           color='lime', fill=False, linewidth=1.2)
        ax.add_patch(circ)
        ax.annotate(s.star_id.replace('HIP_', ''),
                    (s.x_px, s.y_px), color='lime', fontsize=6,
                    xytext=(5, 5), textcoords='offset points')

    # Detected candidates — red crosses
    if label != 'Raw':
        for d in detections:
            ax.plot(d.x, d.y, 'r+', markersize=10, markeredgewidth=1.5)

    ax.set_title(label, fontsize=9)
    ax.axis('off')

# Legend
gt_patch = mpatches.Patch(color='lime', label='Ground-truth (green circle)')
det_patch = mpatches.Patch(color='red',  label='Detection (red cross)')
fig.legend(handles=[gt_patch, det_patch], loc='lower center',
           ncol=2, fontsize=8, framealpha=0.8)

fig.suptitle('Star Detection — Synthetic Data\n'
             f'GT={len(sample_sf.stars)} stars  |  '
             f'Detected={len(detections)}  |  '
             f'seed={sample_sf.seed}',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig('notebooks/fig_05_detections_overlay.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_05_detections_overlay.png')

---
## 4. Centroid accuracy evaluation

In [ ]:
# Match each ground-truth star to its nearest detection
def match_detections(gt_stars, detections, max_dist=3.0):
    """Greedy nearest-neighbour matching between GT and detections."""
    matched, unmatched_gt, false_det = [], [], []
    used = set()
    for gt in gt_stars:
        best_d, best_i = float('inf'), -1
        for i, det in enumerate(detections):
            if i in used:
                continue
            d = math.hypot(det.x - gt.x_px, det.y - gt.y_px)
            if d < best_d:
                best_d, best_i = d, i
        if best_d <= max_dist:
            matched.append((gt, detections[best_i], best_d))
            used.add(best_i)
        else:
            unmatched_gt.append(gt)
    for i, det in enumerate(detections):
        if i not in used:
            false_det.append(det)
    return matched, unmatched_gt, false_det

matched, unmatched_gt, false_det = match_detections(sample_sf.stars, detections)

print(f'Matched     : {len(matched)}/{len(sample_sf.stars)}')
print(f'Missed (FN) : {len(unmatched_gt)}')
print(f'False (FP)  : {len(false_det)}')

if matched:
    errors = [(m[2], m[1].x - m[0].x_px, m[1].y - m[0].y_px) for m in matched]
    print()
    print('Per-star centroid errors:')
    for dist, dx, dy in errors:
        print(f'  dist={dist:.3f} px   Δx={dx:+.3f}   Δy={dy:+.3f}')
    mean_err = np.mean([e[0] for e in errors])
    print(f'\nMean centroid error: {mean_err:.3f} px')

---
## 5. Threshold sensitivity comparison

In [ ]:
thresholds = [0.02, 0.05, 0.10, 0.15, 0.20]
rows = []
for t in thresholds:
    cfg_t = {**det_cfg, 'min_brightness': t, 'min_peak_brightness': t * 0.8}
    dets = detect_stars(normed, cfg_t)
    matched_t, fn_t, fp_t = match_detections(sample_sf.stars, dets)
    rows.append({
        'threshold': t,
        'n_detected': len(dets),
        'n_matched': len(matched_t),
        'n_missed': len(fn_t),
        'n_false': len(fp_t),
    })

import pandas as pd
df_thresh = pd.DataFrame(rows)
print(df_thresh.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(df_thresh['threshold'], df_thresh['n_matched'],  'o-', color='seagreen', label='Matched (TP)')
ax.plot(df_thresh['threshold'], df_thresh['n_missed'],   's--', color='tomato',   label='Missed (FN)')
ax.plot(df_thresh['threshold'], df_thresh['n_false'],    '^:', color='steelblue', label='False (FP)')
ax.axvline(det_cfg['min_brightness'], color='gray', linestyle=':', alpha=0.7, label='Config threshold')
ax.set_xlabel('Threshold')
ax.set_ylabel('Count')
ax.set_title('Detection vs. Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(df_thresh['threshold'].astype(str), df_thresh['n_detected'],
       color='cornflowerblue', edgecolor='white')
ax.axhline(len(sample_sf.stars), color='tomato', linestyle='--',
           label=f'GT n_stars={len(sample_sf.stars)}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Detections')
ax.set_title('Total Detections per Threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('notebooks/fig_06_threshold_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_06_threshold_sensitivity.png')

---
## 6. Batch detection statistics

In [ ]:
# Generate 50 frames, run full preprocess + detect on each
batch_size = 50
results = []

for seed in range(batch_size):
    sf = generator.generate(seed=seed)

    # Preprocess array in-memory (no disk I/O)
    from src.preprocessing.image_preprocessing import subtract_background, reduce_noise, normalise
    img = subtract_background(sf.image,
                               method=pp_cfg['background_method'],
                               filter_size=pp_cfg['background_filter_size'])
    img = reduce_noise(img, method=pp_cfg['noise_method'], sigma=pp_cfg['noise_sigma'])
    img = normalise(img, method=pp_cfg['normalization'])

    dets = detect_stars(img, det_cfg)

    # Match GT vs detections
    matched_b, fn_b, fp_b = match_detections(sf.stars, dets)
    results.append({
        'seed':       seed,
        'n_gt':       len(sf.stars),
        'n_det':      len(dets),
        'n_matched':  len(matched_b),
        'n_missed':   len(fn_b),
        'n_false':    len(fp_b),
        'mean_err_px': np.mean([m[2] for m in matched_b]) if matched_b else float('nan'),
    })

df_batch = pd.DataFrame(results)

n_with_stars = (df_batch['n_gt'] > 0).sum()
tp = df_batch['n_matched'].sum()
fn = df_batch['n_missed'].sum()
fp = df_batch['n_false'].sum()
dr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
fpr = fp / batch_size
mean_ce = df_batch['mean_err_px'].dropna().mean()

print(f'Batch summary ({batch_size} images):')
print(f'  Frames with ≥1 GT star : {n_with_stars}')
print(f'  True positives  (TP)   : {tp}')
print(f'  False negatives (FN)   : {fn}')
print(f'  False positives (FP)   : {fp}')
print(f'  Detection rate         : {dr:.3f}')
print(f'  FP per image           : {fpr:.2f}')
print(f'  Mean centroid error    : {mean_ce:.3f} px')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# GT vs detected count scatter
ax = axes[0]
ax.scatter(df_batch['n_gt'], df_batch['n_det'], alpha=0.5, color='steelblue', s=25)
lim = max(df_batch['n_gt'].max(), df_batch['n_det'].max()) + 1
ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfect detection')
ax.set_xlabel('Ground-truth stars')
ax.set_ylabel('Detected stars')
ax.set_title('GT vs Detected Count')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Centroid error distribution
ax = axes[1]
errs = df_batch['mean_err_px'].dropna()
ax.hist(errs, bins=15, color='mediumseagreen', edgecolor='white')
ax.axvline(errs.mean(), color='tomato', linestyle='--',
           label=f'Mean = {errs.mean():.3f} px')
ax.set_xlabel('Mean centroid error (px)')
ax.set_ylabel('Frames')
ax.set_title('Centroid Error Distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# TP / FN / FP stacked bar per frame (first 20 frames for readability)
ax = axes[2]
sub = df_batch.head(20)
x = np.arange(len(sub))
ax.bar(x, sub['n_matched'], label='TP', color='seagreen')
ax.bar(x, sub['n_missed'], bottom=sub['n_matched'], label='FN', color='tomato')
ax.bar(x, sub['n_false'],  bottom=sub['n_matched'] + sub['n_missed'],
       label='FP', color='steelblue', alpha=0.7)
ax.set_xlabel('Frame index')
ax.set_ylabel('Star count')
ax.set_title('TP / FN / FP per Frame (first 20)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('notebooks/fig_07_batch_statistics.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_07_batch_statistics.png')

---
## 7. Pixel intensity profile through a star

In [ ]:
# Take the brightest detection and plot a horizontal intensity slice
if detections:
    d = detections[0]   # brightest
    row = int(round(d.y))
    row = max(0, min(normed.shape[0] - 1, row))

    # 30-pixel window around the star
    w = 15
    c_start = max(0, int(d.x) - w)
    c_end   = min(normed.shape[1], int(d.x) + w + 1)
    cols    = np.arange(c_start, c_end)

    raw_slice    = raw[row, c_start:c_end]
    normed_slice = normed[row, c_start:c_end]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(cols, raw_slice,    'b-',  label='Raw', linewidth=1.5)
    ax.plot(cols, normed_slice, 'r-',  label='Preprocessed', linewidth=1.5)
    ax.axvline(d.x, color='gray', linestyle=':', alpha=0.8, label=f'Detection x={d.x:.2f}')
    ax.axhline(det_cfg['min_brightness'], color='orange', linestyle='--',
               label=f'Threshold={det_cfg["min_brightness"]}')
    ax.set_xlabel('Column (px)')
    ax.set_ylabel('Intensity')
    ax.set_title(f'Horizontal intensity slice through brightest detection (row={row})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/fig_08_intensity_profile.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Figure saved → notebooks/fig_08_intensity_profile.png')
else:
    print('No detections in sample frame — skipping intensity profile plot.')

---
## 8. Phase 3 preparation notes

**Phase 2 summary:**

| Item | Status |
|---|---|
| Background subtraction (median filter, size 31) | ✅ Implemented |
| Noise reduction (Gaussian blur, σ=0.8 px) | ✅ Implemented |
| Normalisation (min-max, p99.9) | ✅ Implemented |
| Full `preprocess()` pipeline (load → bgsub → denoise → normalise) | ✅ Implemented |
| Star detection (threshold + connected components + centroiding) | ✅ Implemented |
| StarCandidate (x, y, brightness, peak, area, bbox) | ✅ Implemented |
| Unit tests (60 assertions, 9 test classes) | ✅ Passing |

**Observations for Phase 3 (Neural Network):**

1. **Sparse catalog**: Many frames yield 0–2 stars because the prototype catalog has only 50 stars. Before Phase 3 training, extend the catalog to the full Hipparcos dataset (~118K stars) so every pointing has a realistic star density (typically 5–20 stars in a 20° FoV).

2. **Centroid accuracy**: The intensity-weighted centroid achieves < 0.5 px error on noiseless synthetic data. With noise (shot + read), error is slightly higher — still well within the 1.5 px tolerance. This is sufficient for Phase 3 feature extraction.

3. **False positives**: At the current threshold, some noise peaks are detected as star candidates (especially in frames with 0 GT stars). A minimum-area filter (already in place) suppresses most, but the `min_peak_brightness` threshold may need tuning once the catalog is extended and star densities are realistic.

4. **Feature extraction design (Phase 3)**: The `extract_features()` stub in `star_detection.py` will accept a `list[StarCandidate]` and return an array. Candidates for the representation include pairwise angular distances between centroids (scale/rotation invariant) and brightness ratios.

**Next notebook:** `03_pattern_generation.ipynb` — training data generation and feature extraction.